In [1]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC — regional_factors anchor sweep
# Goal: locate '% of KZN rate' on CI Basis + Spouse Basis,
# dump the surrounding cells so we can lock in extraction offsets.
# ─────────────────────────────────────────────────────────────
from pathlib import Path
from openpyxl import load_workbook

BASE   = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE = BASE / "Basis and Assumptions.xlsx"

assert SOURCE.exists(), f"Source not found: {SOURCE}"

wb = load_workbook(SOURCE, data_only=True, read_only=False)

def find_anchors(ws, text):
    """Return (row, col) tuples for cells whose stripped str value == text."""
    hits = []
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == text:
                hits.append((cell.row, cell.column))
    return hits

def dump_block(ws, r0, c0, n_rows=14, n_cols=3, label=""):
    print(f"\n── {label}  anchor at (row={r0}, col={c0})  sheet='{ws.title}' ──")
    print(f"{'row':>4} | " + " | ".join(f"col{c0+i:>3}" for i in range(n_cols)))
    print("-" * (7 + n_cols * 12))
    for dr in range(n_rows):
        r = r0 + dr
        vals = []
        for dc in range(n_cols):
            v = ws.cell(row=r, column=c0 + dc).value
            if v is None:
                vals.append("·")
            elif isinstance(v, float):
                vals.append(f"{v:.4g}")
            else:
                vals.append(str(v)[:10])
        print(f"{r:>4} | " + " | ".join(f"{s:>10}" for s in vals))

# ── CI Basis ──────────────────────────────────────────────────
ws_ci = wb["CI Basis"]
ci_hits = find_anchors(ws_ci, "% of KZN rate")
print(f"CI Basis    — '% of KZN rate' hits: {ci_hits}")
for i, (r, c) in enumerate(ci_hits, 1):
    dump_block(ws_ci, r, c - 1, n_rows=14, n_cols=3, label=f"CI Basis #{i}")

# ── Spouse Basis ──────────────────────────────────────────────
ws_sp = wb["Spouse Basis"]
sp_hits = find_anchors(ws_sp, "% of KZN rate")
print(f"\nSpouse Basis — '% of KZN rate' hits: {sp_hits}")
for i, (r, c) in enumerate(sp_hits, 1):
    dump_block(ws_sp, r, c - 1, n_rows=14, n_cols=3, label=f"Spouse Basis #{i}")

# ── Cross-check: any other anchor variants worth knowing? ─────
for sheet_name in ["CI Basis", "Spouse Basis"]:
    ws = wb[sheet_name]
    variants = ["KZN", "Kwazulu/Natal", "Gauteng", "Regional factors"]
    print(f"\n[{sheet_name}] variant probe:")
    for v in variants:
        hits = find_anchors(ws, v)
        if hits:
            print(f"  {v!r:>20} -> {hits}")

CI Basis    — '% of KZN rate' hits: [(69, 14)]

── CI Basis #1  anchor at (row=69, col=13)  sheet='CI Basis' ──
 row | col 13 | col 14 | col 15
-------------------------------------------
  69 |          · | % of KZN r |          ·
  70 | Kwazulu/Na |          1 |          ·
  71 | Mpumalanga |       0.86 |          ·
  72 |    Gauteng |       0.59 |          ·
  73 | North West |       0.68 |          ·
  74 |    Limpopo |        0.4 |          ·
  75 |        OFS |       0.77 |          ·
  76 | Eastern Ca |       0.58 |          ·
  77 | Western Ca |       0.16 |          ·
  78 | Northern C |       0.28 |          ·
  79 | Nationwide |       0.58 |          ·
  80 |   eSwatini |       0.93 |          ·
  81 |    Namibia |       0.86 |          ·
  82 |          · |          · |          ·

Spouse Basis — '% of KZN rate' hits: [(16, 14)]

── Spouse Basis #1  anchor at (row=16, col=13)  sheet='Spouse Basis' ──
 row | col 13 | col 14 | col 15
------------------------------------------

In [2]:
# ═══════════════════════════════════════════════════════════════
# build_regional_factors.ipynb — Cell 1: Setup
# ═══════════════════════════════════════════════════════════════
# Builds the 'regional_factors' sheet of the Basis Master.
# Schema: product | region | pct_of_kzn
# Products: ci (12 regions), spouse (11 regions)
# Expected total rows: 23
# ═══════════════════════════════════════════════════════════════

import csv
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import polars as pl
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.table import Table, TableStyleInfo

# ── Paths ──────────────────────────────────────────────────────
BASE          = Path(r"J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine")
SOURCE        = BASE / "Basis and Assumptions.xlsx"
MASTER_XLSX   = BASE / "basis_master" / "basis_master_v2026_07.xlsx"
SNAPSHOT_DIR  = BASE / "basis_snapshots" / "basis_v2026_07"
BUILD_LOG     = SNAPSHOT_DIR / "build_log.csv"

# ── Sheet identity ─────────────────────────────────────────────
SHEET_NAME    = "regional_factors"
PARQUET_PATH  = SNAPSHOT_DIR / f"{SHEET_NAME}.parquet"
BASIS_VERSION = "B2026_07"
BUILT_BY      = "Mohamed Hussein Gadiwala"

# ── Ensure output dirs exist ───────────────────────────────────
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
MASTER_XLSX.parent.mkdir(parents=True, exist_ok=True)

# ── Source guard ───────────────────────────────────────────────
assert SOURCE.exists(),        f"Source workbook not found: {SOURCE}"
assert SNAPSHOT_DIR.exists(),  f"Snapshot dir missing:      {SNAPSHOT_DIR}"

# ── Master xlsx open-lock guard (skip if not yet created) ──────
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run."
        )

# ── Sanity print ───────────────────────────────────────────────
print("✅ Cell 1 — Setup complete")
print(f"   Source        : {SOURCE.name}")
print(f"   Master xlsx   : {MASTER_XLSX.name}  (exists: {MASTER_XLSX.exists()})")
print(f"   Snapshot dir  : {SNAPSHOT_DIR}")
print(f"   Target sheet  : {SHEET_NAME}")
print(f"   Basis version : {BASIS_VERSION}")

✅ Cell 1 — Setup complete
   Source        : Basis and Assumptions.xlsx
   Master xlsx   : basis_master_v2026_07.xlsx  (exists: True)
   Snapshot dir  : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07
   Target sheet  : regional_factors
   Basis version : B2026_07


In [3]:
# ═══════════════════════════════════════════════════════════════
# build_regional_factors.ipynb — Cell 2: Helpers
# ═══════════════════════════════════════════════════════════════
# Reusable utilities for anchor-walk extraction, safe coercion,
# and file hashing. Names deliberately avoid IPython-reserved
# identifiers (_i, _s, _f, _, __).
# ═══════════════════════════════════════════════════════════════

def find_anchors(ws, text):
    """
    Return list of (row, col) tuples for cells whose stripped
    string value exactly matches `text`. Walk order: top-to-bottom,
    then left-to-right (openpyxl iter_rows default).
    """
    hits = []
    target = text.strip()
    for row in ws.iter_rows():
        for cell in row:
            v = cell.value
            if isinstance(v, str) and v.strip() == target:
                hits.append((cell.row, cell.column))
    return hits


def as_str(v):
    """Coerce to stripped string; empty/None -> None."""
    if v is None:
        return None
    s = str(v).strip()
    return s if s else None


def as_float(v):
    """Coerce to float; None/blank/non-numeric -> None."""
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def as_int(v):
    """Coerce to int; None/blank/non-numeric -> None."""
    if v is None:
        return None
    if isinstance(v, int) and not isinstance(v, bool):
        return v
    if isinstance(v, float):
        return int(v) if v.is_integer() else None
    s = str(v).strip()
    if not s:
        return None
    try:
        return int(s)
    except ValueError:
        try:
            f = float(s)
            return int(f) if f.is_integer() else None
        except ValueError:
            return None


def sha256_of(path):
    """Return hex SHA-256 of a file's contents."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()


def utc_now_iso():
    """ISO 8601 UTC timestamp, second precision, no microseconds."""
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


# ── Sanity print ───────────────────────────────────────────────
print("✅ Cell 2 — Helpers loaded")
print(f"   find_anchors, as_str, as_float, as_int, sha256_of, utc_now_iso")
print(f"   Reserved-name guard: NOT using _i / _s / _f / _")

✅ Cell 2 — Helpers loaded
   find_anchors, as_str, as_float, as_int, sha256_of, utc_now_iso
   Reserved-name guard: NOT using _i / _s / _f / _


In [4]:
# ═══════════════════════════════════════════════════════════════
# build_regional_factors.ipynb — Cell 3: SPECS
# ═══════════════════════════════════════════════════════════════
# One dict per product. Anchor coords confirmed via diagnostic
# sweep — see handover Appendix A.
#
# Layout (both products, identical shape):
#   anchor cell     : "% of KZN rate"     (header)
#   region column   : anchor_col - 1
#   factor column   : anchor_col          (same col as anchor)
#   data starts     : anchor_row + 1
#   data ends       : first blank region cell (walk until None)
# ═══════════════════════════════════════════════════════════════

SPECS = [
    {
        "product":       "ci",
        "sheet":         "CI Basis",
        "anchor_text":   "% of KZN rate",
        "anchor_occ":    1,       # 1-indexed occurrence
        "region_offset": -1,      # cols left of anchor
        "factor_offset":  0,      # same col as anchor
        "expected_rows": 12,      # KZN..Namibia (see diagnostic)
    },
    {
        "product":       "spouse",
        "sheet":         "Spouse Basis",
        "anchor_text":   "% of KZN rate",
        "anchor_occ":    1,
        "region_offset": -1,
        "factor_offset":  0,
        "expected_rows": 11,      # KZN..Namibia, no eSwatini
    },
]

# ── Load workbook once, share across all specs ─────────────────
wb = load_workbook(SOURCE, data_only=True, read_only=False)

# ── Validate every anchor resolves BEFORE we extract ───────────
print("🔍 Anchor validation:")
for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    occ = spec["anchor_occ"]
    assert len(hits) >= occ, (
        f"[{spec['product']}] expected ≥{occ} '{spec['anchor_text']}' "
        f"anchor(s) on '{spec['sheet']}', found {len(hits)}: {hits}"
    )
    r, c = hits[occ - 1]
    print(f"   {spec['product']:<8} → {spec['sheet']:<14} "
          f"anchor #{occ} at (row={r}, col={c})  "
          f"region@col{c + spec['region_offset']}  factor@col{c + spec['factor_offset']}")

print("\n✅ Cell 3 — SPECS loaded, all anchors resolved")

🔍 Anchor validation:
   ci       → CI Basis       anchor #1 at (row=69, col=14)  region@col13  factor@col14
   spouse   → Spouse Basis   anchor #1 at (row=16, col=14)  region@col13  factor@col14

✅ Cell 3 — SPECS loaded, all anchors resolved


In [5]:
# ═══════════════════════════════════════════════════════════════
# build_regional_factors.ipynb — Cell 4: Extract
# ═══════════════════════════════════════════════════════════════
# Walk each anchor downward until we hit a blank region cell.
# Build all_rows as list of dicts, then assemble polars DataFrame.
# Schema: product | region | pct_of_kzn
# ═══════════════════════════════════════════════════════════════

all_rows = []

for spec in SPECS:
    ws = wb[spec["sheet"]]
    hits = find_anchors(ws, spec["anchor_text"])
    anchor_row, anchor_col = hits[spec["anchor_occ"] - 1]

    region_col = anchor_col + spec["region_offset"]
    factor_col = anchor_col + spec["factor_offset"]

    # Walk downward from the row after the anchor
    r = anchor_row + 1
    rows_extracted = 0
    max_walk = 50   # safety guard — no basis has >50 regions

    while r < anchor_row + 1 + max_walk:
        region = as_str(ws.cell(row=r, column=region_col).value)
        factor = as_float(ws.cell(row=r, column=factor_col).value)

        # Stop condition: blank region cell = end of table
        if region is None:
            break

        all_rows.append({
            "product":     spec["product"],
            "region":      region,
            "pct_of_kzn":  factor,
        })
        rows_extracted += 1
        r += 1

    print(f"   {spec['product']:<8} → {rows_extracted:>3} rows extracted "
          f"(expected {spec['expected_rows']})")

# ── Assemble DataFrame ─────────────────────────────────────────
df = pl.DataFrame(all_rows, infer_schema_length=None)

print(f"\n✅ Cell 4 — Extract complete")
print(f"   Total rows        : {df.height}")
print(f"   Distinct products : {df['product'].n_unique()}")
print(f"   Schema            : {df.schema}")
print("\n── Preview (first 5 + last 5) ──")
print(df.head(5))
print(df.tail(5))

   ci       →  12 rows extracted (expected 12)
   spouse   →  11 rows extracted (expected 11)

✅ Cell 4 — Extract complete
   Total rows        : 23
   Distinct products : 2
   Schema            : Schema([('product', String), ('region', String), ('pct_of_kzn', Float64)])

── Preview (first 5 + last 5) ──
shape: (5, 3)
┌─────────┬───────────────┬────────────┐
│ product ┆ region        ┆ pct_of_kzn │
│ ---     ┆ ---           ┆ ---        │
│ str     ┆ str           ┆ f64        │
╞═════════╪═══════════════╪════════════╡
│ ci      ┆ Kwazulu/Natal ┆ 1.0        │
│ ci      ┆ Mpumalanga    ┆ 0.86       │
│ ci      ┆ Gauteng       ┆ 0.59       │
│ ci      ┆ North West    ┆ 0.68       │
│ ci      ┆ Limpopo       ┆ 0.4        │
└─────────┴───────────────┴────────────┘
shape: (5, 3)
┌─────────┬───────────────┬────────────┐
│ product ┆ region        ┆ pct_of_kzn │
│ ---     ┆ ---           ┆ ---        │
│ str     ┆ str           ┆ f64        │
╞═════════╪═══════════════╪════════════╡
│ spouse  

In [6]:
# ═══════════════════════════════════════════════════════════════
# build_regional_factors.ipynb — Cell 5: Sanity checks
# ═══════════════════════════════════════════════════════════════
# Assertions that would catch silent breakage of the extraction.
# Values sourced from diagnostic sweep — NOT computed, NOT rounded.
# ═══════════════════════════════════════════════════════════════

# ── Structural checks ──────────────────────────────────────────
assert df.height == 23, f"expected 23 rows, got {df.height}"
assert set(df.columns) == {"product", "region", "pct_of_kzn"}, \
    f"unexpected columns: {df.columns}"
assert df["product"].n_unique() == 2, \
    f"expected 2 products, got {df['product'].n_unique()}"

# ── Per-product row counts ─────────────────────────────────────
counts = df.group_by("product").len().sort("product")
ci_rows     = counts.filter(pl.col("product") == "ci")["len"][0]
spouse_rows = counts.filter(pl.col("product") == "spouse")["len"][0]
assert ci_rows == 12,     f"ci     : expected 12 rows, got {ci_rows}"
assert spouse_rows == 11, f"spouse : expected 11 rows, got {spouse_rows}"

# ── No null regions (structural integrity) ─────────────────────
assert df["region"].null_count() == 0, "found null region(s)"

# ── Value spot-checks (from diagnostic dump) ───────────────────
SPOT_CHECKS = [
    # (product, region, expected pct_of_kzn)
    ("ci",     "Kwazulu/Natal",  1.00),
    ("ci",     "Mpumalanga",     0.86),
    ("ci",     "Gauteng",        0.59),
    ("ci",     "North West",     0.68),
    ("ci",     "Limpopo",        0.40),
    ("ci",     "OFS",            0.77),
    ("ci",     "Eastern Cape",   0.58),
    ("ci",     "Western Cape",   0.16),
    ("ci",     "Northern Cape",  0.28),
    ("ci",     "Nationwide",     0.58),
    ("ci",     "eSwatini",       0.93),
    ("ci",     "Namibia",        0.86),
    # Spouse: all zeros in source (flagged for actuarial review, NOT rewritten)
    ("spouse", "Kwazulu/Natal",  0.00),
    ("spouse", "Namibia",        0.00),
]

TOL = 1e-6
print("🎯 Spot checks:")
fails = 0
for product, region, expected in SPOT_CHECKS:
    got = df.filter(
        (pl.col("product") == product) & (pl.col("region") == region)
    )["pct_of_kzn"][0]
    ok = got is not None and abs(got - expected) < TOL
    flag = "✅" if ok else "❌"
    print(f"   {flag}  {product:<8} {region:<15} expected={expected:<6}  got={got}")
    if not ok:
        fails += 1

assert fails == 0, f"{fails} spot check(s) failed"

# ── eSwatini appears ONLY in ci (per spec) ─────────────────────
eswatini_products = set(
    df.filter(pl.col("region") == "eSwatini")["product"].to_list()
)
assert eswatini_products == {"ci"}, \
    f"eSwatini should only appear for ci, got {eswatini_products}"

# ── Advisory: flag if any Spouse factor != 0 (would be a surprise) ──
spouse_nonzero = df.filter(
    (pl.col("product") == "spouse") & (pl.col("pct_of_kzn") != 0.0)
)
if spouse_nonzero.height > 0:
    print(f"\n⚠️  ADVISORY: {spouse_nonzero.height} Spouse row(s) with non-zero factor:")
    print(spouse_nonzero)
else:
    print("\n📌 Note: all Spouse pct_of_kzn = 0 in source (flagged for actuarial review).")

print("\n✅ Cell 5 — All sanity checks passed")

🎯 Spot checks:
   ✅  ci       Kwazulu/Natal   expected=1.0     got=1.0
   ✅  ci       Mpumalanga      expected=0.86    got=0.86
   ✅  ci       Gauteng         expected=0.59    got=0.59
   ✅  ci       North West      expected=0.68    got=0.68
   ✅  ci       Limpopo         expected=0.4     got=0.4
   ✅  ci       OFS             expected=0.77    got=0.77
   ✅  ci       Eastern Cape    expected=0.58    got=0.58
   ✅  ci       Western Cape    expected=0.16    got=0.16
   ✅  ci       Northern Cape   expected=0.28    got=0.28
   ✅  ci       Nationwide      expected=0.58    got=0.58
   ✅  ci       eSwatini        expected=0.93    got=0.9299999999999999
   ✅  ci       Namibia         expected=0.86    got=0.86
   ✅  spouse   Kwazulu/Natal   expected=0.0     got=0.0
   ✅  spouse   Namibia         expected=0.0     got=0.0

📌 Note: all Spouse pct_of_kzn = 0 in source (flagged for actuarial review).

✅ Cell 5 — All sanity checks passed


In [7]:
# ═══════════════════════════════════════════════════════════════
# build_regional_factors.ipynb — Cell 6: Write outputs + build log
# ═══════════════════════════════════════════════════════════════
# 1) Write parquet snapshot
# 2) Upsert sheet in master xlsx (delete-if-exists, create, style,
#    format as Excel Table)
# 3) Append audit row to build_log.csv
# ═══════════════════════════════════════════════════════════════

# ── 1) Parquet ─────────────────────────────────────────────────
df.write_parquet(PARQUET_PATH)
parquet_hash = sha256_of(PARQUET_PATH)
parquet_kb   = round(PARQUET_PATH.stat().st_size / 1024, 2)
print(f"📦 Parquet written : {PARQUET_PATH.name}  ({parquet_kb} KB)")

# ── 2) Master xlsx ─────────────────────────────────────────────
# Re-run the open-lock guard immediately before write
if MASTER_XLSX.exists():
    try:
        with open(MASTER_XLSX, "a"):
            pass
    except PermissionError:
        raise PermissionError(
            f"'{MASTER_XLSX.name}' is open in Excel. Close it and re-run."
        )

# Open (or create) master
if MASTER_XLSX.exists():
    wb_master = load_workbook(MASTER_XLSX)
else:
    from openpyxl import Workbook
    wb_master = Workbook()
    # remove the default empty sheet only if it's the only one
    if "Sheet" in wb_master.sheetnames and len(wb_master.sheetnames) == 1:
        del wb_master["Sheet"]

# Delete old sheet if present
if SHEET_NAME in wb_master.sheetnames:
    del wb_master[SHEET_NAME]

ws_new = wb_master.create_sheet(SHEET_NAME)

# Write header + data
columns = df.columns
ws_new.append(columns)
for row in df.iter_rows():
    ws_new.append(list(row))

# Style header row (avoid ws[1] bracket-render issue)
header_row = next(ws_new.iter_rows(min_row=1, max_row=1))
for cell in header_row:
    cell.font      = Font(bold=True, color="FFFFFF")
    cell.fill      = PatternFill("solid", fgColor="305496")
    cell.alignment = Alignment(horizontal="left")

# Auto-size column widths (rough heuristic)
for col_idx, col_name in enumerate(columns, start=1):
    max_len = max(
        [len(str(col_name))] +
        [len(str(v)) if v is not None else 0 for v in df[col_name].to_list()]
    )
    ws_new.column_dimensions[get_column_letter(col_idx)].width = max(12, max_len + 2)

# Format as Excel Table
last_col_letter = get_column_letter(len(columns))
table_ref = f"A1:{last_col_letter}{df.height + 1}"
tbl = Table(displayName=f"tbl_{SHEET_NAME}", ref=table_ref)
tbl.tableStyleInfo = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False,
)
ws_new.add_table(tbl)

wb_master.save(MASTER_XLSX)
print(f"📗 Master xlsx     : {MASTER_XLSX.name} — sheet '{SHEET_NAME}' upserted")

# ── 3) Build log ───────────────────────────────────────────────
source_hash = sha256_of(SOURCE)

log_row = {
    "timestamp_utc":     utc_now_iso(),
    "sheet":             SHEET_NAME,
    "row_count":         df.height,
    "distinct_products": df["product"].n_unique(),
    "source_file":       SOURCE.name,
    "source_sha256":     source_hash,
    "parquet_file":      PARQUET_PATH.name,
    "parquet_sha256":    parquet_hash,
    "parquet_kb":        parquet_kb,
    "status":            "ok",
    "built_by":          BUILT_BY,
}

log_exists = BUILD_LOG.exists()
with open(BUILD_LOG, "a", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(log_row.keys()))
    if not log_exists:
        writer.writeheader()
    writer.writerow(log_row)

print(f"📝 Build log       : appended row for '{SHEET_NAME}'")
print(f"   source_sha256   : {source_hash[:16]}…")
print(f"   parquet_sha256  : {parquet_hash[:16]}…")

print(f"\n✅ Cell 6 — regional_factors build COMPLETE")
print(f"   Rows            : {df.height}")
print(f"   Products        : {sorted(df['product'].unique().to_list())}")
print(f"   Parquet         : {PARQUET_PATH}")
print(f"   Master sheet    : {SHEET_NAME} in {MASTER_XLSX.name}")

📦 Parquet written : regional_factors.parquet  (1.35 KB)
📗 Master xlsx     : basis_master_v2026_07.xlsx — sheet 'regional_factors' upserted
📝 Build log       : appended row for 'regional_factors'
   source_sha256   : 10d700d0c785719c…
   parquet_sha256  : c04171e743bd6987…

✅ Cell 6 — regional_factors build COMPLETE
   Rows            : 23
   Products        : ['ci', 'spouse']
   Parquet         : J:\Product & Client Solutions\Toolbox\Calculators\Business Tools\Tender Summary\GAP Pricing Engine\basis_snapshots\basis_v2026_07\regional_factors.parquet
   Master sheet    : regional_factors in basis_master_v2026_07.xlsx
